# 07 Тренд-таблица_Статистическая значимость #

In [ ]:
import pandas as pd
from functools import reduce
from brandpulse_api.calc import BrandpulseCalc
from brandpulse_api.upload import BrandpulseUpload

# имя файла с данными
DB_FILE = 'brandpulse3.duckdb'

# создаем объект для расчетов
bp = BrandpulseCalc(db_file=DB_FILE)

# создаем объект для загрузки
upload = BrandpulseUpload(db_file=DB_FILE)

Исследуем доступные проекты

In [ ]:
# вывод всех проектов
#bp.find_projects()

# поиск по подстроке в имени проекта
# bp.find_projects(project_name='желуд')

# поиск по двум возможным вариантам имени проекта
# bp.find_projects(project_name='желуд|майон')

# поиск по подстроке в английском имени проекта
#bp.find_projects(project_name_eng='gastro')

# поиск по системному имени категории
#bp.find_projects(category_sys_name='JEL')

# поиск по имени категории
#bp.find_projects(category_name='желуд')

# поиск по английскому имени категории
#bp.find_projects(category_name_eng='gastro')

# поиск по дате начала волны 
#bp.find_projects(wave_begin='2025-09-01')

# поиск по дате конца волны 
#bp.find_projects(wave_end='2025-09-28')

# поиск по имени волны 
#bp.find_projects(wave_name="Сентябрь'25")

# поиск проекта по нескольким параметрам
bp.find_projects(project_name='Обезболивающие', wave_name="24|25")


Исследуем бренды

In [ ]:
# вывод всех брендов
#bp.find_brands()

# вывод по имени бренда
bp.find_brands(brand_name=r'Найз')

# поиск бренда по нескольким параметрам - категории и названию 
#bp.find_brands(brand_name='LIPTON', category_name='Чай|холодный чай')

Исследуем существующие комбинации категории-бренды-свойства из ответов всех загруженных)

In [ ]:
# вывод всех комбинаций
#bp.find_category_brand_property()

# вывод комбинаций для категории Сосиски
#bp.find_category_brand_property(category_name='Сосиски')

# вывод комбинаций для нужного свойста
#bp.find_category_brand_property(property_name='Цель поездок: за границу')

# вывод комбинаций для нужной категории
bp.find_category_brand_property(category_name=r'Обезболивающие')


Пример передачи найденных ид проектов и их описаний для загрузки

In [ ]:
# ищем проекты, как выше, но сохряняем их в переменную
prj = bp.find_projects(project_name='Обезболивающие', wave_name="24|25")

# выводим найденные имена для наглядности
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    print(prj['project_name'])
    
# сохраняем описание проекта для вывода далее в шапке файла
description = prj['description'].iloc[0]
print(description)

# отбираем все значения в столбце project_id (на всякий случай сохраняем только уникальные значения)
prj_ids = prj['project_id'].unique().tolist()
prj_ids

In [ ]:
# загружаем найденные проекты
#upload.upload_projects(project_list=prj_ids)

Задаем параметры расчета

In [ ]:
trends = [
    {"name": "Апрель_24", "id": 1955665618766571392}, 
    {"name": "Май_24", "id": 1307147367437962368},
    {"name": "Июнь_24", "id": 4549739308487795584},
    {"name": "Июль_24", "id": 82168593775556416},
    {"name": "Август_24", "id": 5270315564109146176},
    {"name": "Сентябрь_24", "id": 5270315758355140224},
    {"name": "Октябрь_24", "id": 2964472909584691200},
    {"name": "Ноябрь_24", "id": 6135007241091149120},
    {"name": "Декабрь_24", "id": 7071756117135325184},
    {"name": "Январь_25", "id": 3468876519813574080},
    {"name": "Февраль_25", "id": 1739494479366599744},
    {"name": "Март_25", "id": 4621798428647960640}   
    ]

# задаем параметры целевой группы (используя sys_name из базы)
demo = [
    {"property_name": "SEX", 
     "expression": "='FEMALE'"},
]

Справочно исследуем свойства

In [ ]:
# поиск по описанию
# bp.find_property(text='Пол')

# поиск по имени
# bp.find_property(sys_name='NPS_SEG')

# поиск по комбинации имени и описания
# bp.find_property(text='Возраст', sys_name='SEX')

Справочно исследуем значения свойств

In [ ]:
# поиск по описанию
#bp.find_property_options(text='Покупали кофе')

# поиск по имени
#bp.find_property_options(sys_name='ORDERED_FOOD_TO')

# поиск по комбинации имени и описания
#bp.find_property_options(text='кофе', sys_name='COFFEE')

# поиск по имени свойства (символ ^ добавлен для поиска имен, начинающихся с этой строки)
bp.find_property_options(property_sys_name='^GDE_VID_REK')

Рассчитаем universe

In [ ]:
# список свойств c их значением и требуемыми категориями
properties = [
    ('SAA_', ['YES'], ['ASPIRINUPS97','COLDREX95','MOTILIUM63','PANADOL93','RENEWAL59','SOLPADEINE32','ALMAGEL91','AMBROBENE94','AMELOTEKSG97','ANALGIN2',
                       'ANALGINUL3','ANAFERON4','ANVIMAKS5','ANTIGRIPPI10','ARBIDOL13','ASKOFENULT88','ASKOFENP20','ASPIRIN22','ATSETILSAL31','AERTAL37',
                       'BARALGIN53','BETADIN62','BRAL91','BYSTRUMGEL2','VOLTAREN10','VOLTAR_G32','GASTAL45','GEVISKON46','GLITSIN65','DEKSALGIN93','DEKSONAL94',
                       'DETRALEKS2','DIALRAPID4','DIKLOFENAK7','DIMEKSID70','DOLGIT21','DOLOBENE22','DROTAVERIN','DYUSPATALI65','DYUFALAK44','ZODAK64',
                       'ZOLOTAYAZV66','IBUKLIN74','IBUPROFEN75','KAVINTON1','KAPSIKAM2','KETANOV','KETONAL15','KETOPROFEN24','KETOROL01','KODELAK24',
                       'KOLDAKT25','MEZIM36','MEKSIDOL98','MELOXICAM22','MENOVAZIN10','MIG40046','MILDRONAT51','MIRAMISTIN54','MOVALIS61','MOTRIN79',
                       'NISENIMESULIDE','NAJZGEL89','NALGESIN22','NEKST1','NEOBUT20','NIMESIL','NIMESULID','NIMULID52','NOVIGAN6','NOLPAZA49','NOOPEPT10',
                       'NOSHPA11','NOSHPAFOR85','NUROFEN95','NUROFENEXP1','OMEZ27','PANANGIN50','PANCREATINUM20','PARATSETAM52','PENTALGIN57','PENTALGING58',
                       'POLISORB83','RENGALIN20','RINZA31','RINIKOLD82','SEDALGINP84','SMEKTA14','SNUP18','SPAZGAN29','SPAZGANNEO84','SPAZMALGON30',
                       'SPAZMALIN86','SUPRASTIN47','SUPRASTINE48','TEMPALGIN68','TENOTEN69','TERAFLEKS71','TERAFLYU72','TRIGAND99','TRIMEDAT96','UPSARINUPS7',
                       'FASTUMGEL22','FERVEKS24','FESTAL25','FINALGON33','KHONDROKSI63','TSITRAMON83','EVALAR10','ERGOFERON27','ESPUMIZAN32','EFFERALGAN34',
                       'DRUGOE55','ZATRUDNYAYS']), 
    ('AD_AIDED_AW_', ['YES'], ['SOLPADEINE32','ANALGIN2','ANALGINUL3','ASKOFENP20','ASPIRIN22','ATSETILSAL31','BARALGIN53','DEKSALGIN93','DEKSONAL94',
                               'IBUKLIN74','KETANOV','KETONAL15','KETOROL01','MIG40046','MOTRIN79','NISENIMESULIDE','NALGESIN22','NEKST1','NOVIGAN6',
                               'NOSHPA11','NUROFEN95','PARATSETAM52','PENTALGIN57','SPAZGAN29','SPAZMALGON30','TEMPALGIN68','TSITRAMON83','DRUGOE55',
                               'ZATRUDNYAYS']), 
]

data = []
for t in trends:
  df_tmp = (bp.calc_project_brand_universe(project_ids=[t['id']], demo=demo, properties=properties)
            .reset_index(drop=True)
            .rename(columns={"universe": f"{t['name']}_universe", "sample": f"{t['name']}_sample"}))
  data.append(df_tmp)

# Объединяем по общим колонкам
key_cols = df_tmp.columns.to_list()[:-2]
df = reduce(lambda left, right: pd.merge(left, right, on=key_cols, how='outer', suffixes=('', '_dup')), data)

# сортируем 
df1 = df.sort_values(
    by=['property_sys_name', 'option_sys_name', 'Апрель_24_universe'], 
    ascending=[True, True, False]  
)
    
df1

In [ ]:
# список свойств c их значением 
properties = [
    ('GDE_VID_REK', ['TV','RADIO','PAP','TC','KINO','OB_TRAN','METRO','ULIC','BANNER','VIDEOR','AUDIO','MOB_PRIL','POST_SOC','BLOG','OBZOR','SMSMES','OTHER']), 
]

data = []
for t in trends:
  df_tmp = (bp.calc_project_property_universe(project_ids=[t['id']], demo=demo, properties=properties)
            .reset_index(drop=True)
            .rename(columns={"universe": f"{t['name']}_universe", "sample": f"{t['name']}_sample"}))
  data.append(df_tmp)

# Объединяем по общим колонкам
key_cols = df_tmp.columns.to_list()[:-2]
df = reduce(lambda left, right: pd.merge(left, right, on=key_cols, how='outer', suffixes=('', '_dup')), data)

# сортируем 
df2 = df.sort_values(
    by=['property_sys_name', 'Апрель_24_universe'], 
    ascending=[True, False]  
)

# приводим второй датафрейм к единому виду с первым (добавляем недостающие колонки)
df2.insert(loc=df2.columns.get_loc('property_text') + 1, column='brand_sys_name', value='')
df2.insert(loc=df2.columns.get_loc('brand_sys_name') + 1, column='brand_text', value='')

df2

Рассчитаем тоталы

In [ ]:
# добавляем строку сводных итогов
full = pd.concat([df1, df2], ignore_index=True)

non_num_cols = full.columns[:6]

total_row = {
    non_num_cols[0]: 'total',
    non_num_cols[1]: '',
    non_num_cols[2]: '',
    non_num_cols[3]: '',
    non_num_cols[4]: '',
    non_num_cols[5]: '',
}

for t in trends:
    total_row[t['name']+'_universe'] = bp.calc_universe(project_ids=[t['id']], demo=demo)
    total_row[t['name']+'_sample'] = bp.calc_sample(project_ids=[t['id']], demo=demo)
    total_row[t['name']+'_col%'] = 100
total_row

Добавляем к рассчитанному universe расчет долей (col%)

In [ ]:
# считаем доли
result_tmp = full[non_num_cols].copy()

for t in trends:
    result_tmp[t['name']+'_universe'] = full[t['name']+'_universe']
    result_tmp[t['name']+'_sample'] = full[t['name']+'_sample']
    result_tmp[t['name']+'_col%'] = round(100 * full[t['name']+'_universe'] / total_row[t['name']+'_universe'], 1)

total_row_df = pd.DataFrame([total_row], columns=result_tmp.columns)
result = pd.concat([total_row_df, result_tmp], ignore_index=True)

result

In [ ]:
# маскируем минимальный сэмпл
masked = result.copy()

sample_limit = 5

for t in trends:
    mask_condition = (result[t['name']+'_sample'] < sample_limit) | (result[t['name']+'_sample'].isnull())

    masked[t['name']+'_universe'] = result[t['name']+'_universe'].mask(mask_condition, '~')
    masked[t['name']+'_col%'] = result[t['name']+'_col%'].mask(mask_condition, '~')
    masked[t['name']+'_sample'] = result[t['name']+'_sample'].mask(mask_condition, '~')

masked

In [ ]:
# удаляем колонки с сэмплами
res_total = masked.drop(columns=[t['name'] + '_sample' for t in trends])
res_total

Рассчитываем статистическую значимость

In [ ]:
import numpy as np
from scipy.stats import norm

total = total_row_df.copy()
df = result.copy()

def is_calc(row, months, p1, p2):
    if (total.loc[0, f"{months[0]}_sample"] >= 50 
    and total.loc[0, f"{months[1]}_sample"] >= 50
    and total.loc[0, f"{months[0]}_sample"] * p1 > 5
    and total.loc[0, f"{months[0]}_sample"] * (1 - p1) > 5
    and total.loc[0, f"{months[1]}_sample"] * p2 > 5
    and total.loc[0, f"{months[1]}_sample"] * (1- p2) > 5):
        return True
    else:
        return False

def calc_sign1(row, months):
    col_percent1 = pd.to_numeric(row[f"{months[0]}_universe"], errors='coerce')  
    if pd.notna(col_percent1):
        p1 = col_percent1 / float(total.loc[0, f"{months[0]}_universe"])
    else:
        p1 = '~'
    
    col_percent2 = pd.to_numeric(row[f"{months[1]}_universe"], errors='coerce')
    if pd.notna(col_percent2):
        p2 = col_percent2 / float(total.loc[0, f"{months[1]}_universe"])
    else:
        p2 = '~'
    
    if p1 == '~' or p2 == '~':
        return 0

    if is_calc(row, months, p1, p2):
        p = (row[f"{months[0]}_universe"] + row[f"{months[1]}_universe"]) / sum([total.loc[0, f"{months[0]}_universe"], total.loc[0, f"{months[1]}_universe"]])
        z = abs(p1 - p2) / np.sqrt(p * (1 - p) * sum(1 / x for x in [total.loc[0, f"{months[0]}_sample"], total.loc[0, f"{months[1]}_sample"]]))
        signCalc = round((2 * norm.cdf(z) - 1) * 100, 1)
        return signCalc
    else:
        return 'N/A'

base_month = trends[0]['name']
for t in trends[1:]:
    months = [base_month, t['name']]
    df[t['name']+'_signCalc'] = df.apply(calc_sign1, axis=1, args=(months,))
    base_month = t['name']

# маскируем по размеру сэмпла
df_sign1 = df.copy()
sample_limit = 5

for t in trends[1:]:
    mask_condition = (df[t['name']+'_sample'] < sample_limit) | (df[t['name']+'_sample'].isnull())
    df_sign1[t['name']+'_signCalc'] = df[t['name']+'_signCalc'].mask(mask_condition, '~')

# удаляем лишние колонки
df_sign1.drop(columns=[t['name'] + '_sample' for t in trends], inplace=True)
df_sign1.drop(columns=[t['name'] + '_col%' for t in trends], inplace=True)
universe_cols = [t['name'] + '_universe' for t in trends]
df_sign1.drop(columns=universe_cols[1:], inplace=True)
df_sign1


Сохраняем в excel с форматированием отчета

In [ ]:
# сохранение в эксель с форматированием
import os
from openpyxl.styles import Alignment

subfolder = "excel"  # название подпапки
os.makedirs(subfolder, exist_ok=True)  # создаёт подпапку, если не существует

filepath = os.path.join(subfolder, "07_trendtab_stat.xlsx")

with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
    res_total.to_excel(writer, sheet_name='Расчет', startrow=5, startcol=1, index=False)
    
    # настройка листа экселя
    worksheet = writer.sheets['Расчет']
    
    # Устанавливаем ширину столбцов (учитываем startcol=1)
    worksheet.column_dimensions['B'].width = 20  
    worksheet.column_dimensions['C'].width = 20  
    
    worksheet['A1'] = 'Целевая база: Население [BHT_Обезболивающие, антиспазматические препараты, Россия 100k+, 14-64, лично пользовались за последние 6 месяцев]'
    worksheet['A2'] = 'Целевая группа: Пол.Женщины'
    worksheet['A4'] = 'Total'
    #worksheet['A1'].font = worksheet['A1'].font.copy(size=10)
    
    for cell in worksheet[6]:
        cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
    
    worksheet['B6'] = ''
    worksheet['C6'] = ''
    worksheet['D6'] = ''
    worksheet['E6'] = ''
    worksheet['F6'] = ''
    worksheet['G6'] = ''
    
    worksheet.row_dimensions[6].height = 90
    
    df_sign1.to_excel(writer, sheet_name='Расчет Стат знач-ть', startrow=5, startcol=1, index=False)
    
    # настройка листа экселя
    worksheet = writer.sheets['Расчет Стат знач-ть']
    
    # Устанавливаем ширину столбцов (учитываем startcol=1)
    worksheet.column_dimensions['B'].width = 20  
    worksheet.column_dimensions['C'].width = 20  
    
    worksheet['A1'] = 'Целевая база: Население [BHT_Обезболивающие, антиспазматические препараты, Россия 100k+, 14-64, лично пользовались за последние 6 месяцев]'
    worksheet['A2'] = 'Целевая группа: Пол.Женщины'
    worksheet['A4'] = 'Sign Prev.'
    #worksheet['A1'].font = worksheet['A1'].font.copy(size=10)
    
    for cell in worksheet[6]:
        cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
    
    worksheet['B6'] = ''
    worksheet['C6'] = ''
    worksheet['D6'] = ''
    worksheet['E6'] = ''
    worksheet['F6'] = ''
    worksheet['G6'] = ''
    
    worksheet.row_dimensions[6].height = 90